# Fine-tuning LoRA Qwen3-VL 2B con early stopping

Usa pesi bfloat16 non quantizzati e LoRA sul decoder linguistico.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Esegui il notebook dal repository SHIELD o da una sua sottocartella.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIG_PATH = "configs/experiments/qwen3vl_2b_lora_early_stopping.toml"

from shield.notebook_utils import (
    SYSTEM_PROMPT,
    ensure_dirs,
    evaluate_on_test_set,
    load_config,
    load_hf_dataset,
    load_json_records,
    print_dataset_summary,
    resolve_path,
    save_evaluation_results,
    set_seed,
    XRayDataCollator,
)

config = load_config(CONFIG_PATH, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Config:       {CONFIG_PATH}")
print(f"Run:          {config['run']['name']}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Nessuna GPU disponibile. Verifica CUDA sul server.")

print(f"GPU disponibili: {torch.cuda.device_count()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} - {props.total_memory / 1e9:.1f} GB")

In [ ]:
processed_dir = resolve_path(config, "paths", "processed_dir")
checkpoints_dir = resolve_path(config, "paths", "checkpoints_dir")
evaluation_dir = resolve_path(config, "paths", "evaluation_dir")
model_dir = resolve_path(config, "paths", "model_dir")
ensure_dirs(checkpoints_dir, evaluation_dir)

train_dataset = load_hf_dataset(processed_dir / "train.json")
val_dataset = load_hf_dataset(processed_dir / "val.json")
test_records = load_json_records(processed_dir / "test.json")
print_dataset_summary(train_dataset, val_dataset, test_records)

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ft = config["finetuning"]
peft_cfg = config["peft"]
set_seed(ft.get("seed", 42))

quantization = ft.get("quantization", "none")
bnb_config = None
if quantization == "4bit":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

print(f"Caricamento modello LoRA: {model_dir}")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    str(model_dir),
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

if quantization == "4bit":
    model = prepare_model_for_kbit_training(model)

processor = AutoProcessor.from_pretrained(
    str(model_dir),
    min_pixels=256 * 28 * 28,
    max_pixels=ft.get("max_image_pixels", 1003520),
)

n_llm_layers = model.config.text_config.num_hidden_layers
lora_config = LoraConfig(
    r=peft_cfg["lora_r"],
    lora_alpha=peft_cfg["lora_alpha"],
    lora_dropout=peft_cfg["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=peft_cfg["target_modules"],
    layers_to_transform=list(range(n_llm_layers)),
    layers_pattern="layers",
)
model = get_peft_model(model, lora_config)

trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
total = sum(param.numel() for param in model.parameters())
print(f"Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Parametri totali:    {total:,}")

In [ ]:
data_collator = XRayDataCollator(processor, max_seq_length=config["finetuning"].get("max_length", 2048))

sample_batch = data_collator([train_dataset[i] for i in range(min(2, len(train_dataset)))])
labels = sample_batch["labels"]
unmasked = (labels != -100).sum(dim=1).tolist()
print(f"Token non mascherati per esempio: {unmasked}")
if not all(count > 0 for count in unmasked):
    raise RuntimeError("Label masking non valido: almeno un esempio non contribuisce alla loss.")

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

ft = config["finetuning"]
es = config["early_stopping"]

training_args = SFTConfig(
    output_dir=str(checkpoints_dir),
    num_train_epochs=ft["epochs"],
    per_device_train_batch_size=ft["per_device_train_batch_size"],
    per_device_eval_batch_size=ft["per_device_eval_batch_size"],
    gradient_accumulation_steps=ft["gradient_accumulation_steps"],
    learning_rate=ft["learning_rate"],
    lr_scheduler_type=ft.get("lr_scheduler_type", "cosine"),
    warmup_ratio=ft.get("warmup_ratio", 0.05),
    weight_decay=ft.get("weight_decay", 0.01),
    max_grad_norm=ft.get("max_grad_norm", 1.0),
    bf16=ft.get("bf16", True),
    fp16=ft.get("fp16", False),
    save_strategy=ft.get("save_strategy", "epoch"),
    save_total_limit=ft.get("save_total_limit", 3),
    eval_strategy=ft.get("eval_strategy", "epoch"),
    load_best_model_at_end=ft.get("load_best_model_at_end", True),
    metric_for_best_model=ft.get("metric_for_best_model", "eval_loss"),
    greater_is_better=ft.get("greater_is_better", False),
    logging_steps=ft.get("logging_steps", 10),
    report_to="none",
    gradient_checkpointing=ft.get("gradient_checkpointing", True),
    dataloader_num_workers=ft.get("dataloader_num_workers", 2),
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=es.get("patience", 2),
    early_stopping_threshold=es.get("threshold", 0.0),
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[early_stopping],
)

print(f"Avvio training: {config['run']['name']}")
print(f"Epoch massime: {ft['epochs']} | Early stopping patience: {es.get('patience', 2)}")
trainer.train()

trainer_state_path = checkpoints_dir / "trainer_state.json"
trainer.state.save_to_json(str(trainer_state_path))

final_path = checkpoints_dir / "final"
trainer.save_model(str(final_path))
processor.save_pretrained(str(final_path))
print(f"Trainer state: {trainer_state_path}")
print(f"Modello finale/best model salvato in: {final_path}")

In [ ]:
metrics_path = resolve_path(config, "evaluation", "metrics_file")
qualitative_path = resolve_path(config, "evaluation", "qualitative_file")

results = evaluate_on_test_set(
    model=model,
    processor=processor,
    test_records=test_records,
    label=config["run"]["name"],
    bertscore_model_type=config["evaluation"].get("bertscore_model_type", "xlm-roberta-large"),
    max_new_tokens=config["evaluation"].get("max_new_tokens", 512),
    qualitative_limit=config["evaluation"].get("qualitative_examples", 20),
)
save_evaluation_results(results, metrics_path, qualitative_path)

## Logging MLflow

Esegui questa cella solo se il server MLflow e' acceso.

In [ ]:
import subprocess

metrics_path = resolve_path(config, "evaluation", "metrics_file")
evaluation_dir = resolve_path(config, "paths", "evaluation_dir")
trainer_state_path = checkpoints_dir / "trainer_state.json"
cmd = [
    "uv", "run", "python", "scripts/mlflow/log_run_to_mlflow.py",
    "--config", CONFIG_PATH,
    "--trainer-state", str(trainer_state_path.relative_to(PROJECT_ROOT)),
    "--metrics", str(metrics_path.relative_to(PROJECT_ROOT)),
    "--artifact", str(evaluation_dir.relative_to(PROJECT_ROOT)),
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)